In [ ]:
import json
import os
import shutil
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Image
from roboflow import Roboflow
from ultralytics import YOLO

os.chdir("..")
load_dotenv(Path("config", ".env"))

#### Setup Directory

In [ ]:
pth_settings = "/Users/jack.chan/Library/Application Support/Ultralytics/settings.json"

with open(pth_settings, "r") as file:
    data = json.load(file)

data["datasets_dir"] = os.getcwd()

with open(pth_settings, "w") as file:
    json.dump(data, file)

#### Download Dataset

In [ ]:
rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))
project = rf.workspace("jack-chan-edpdi").project("supermarketscanner")
dataset = project.version(5).download("yolov8")

pth_datasets = Path("datasets")
pth_datasets.mkdir(exist_ok=True)
shutil.move("SupermarketScanner-5", pth_datasets)

#### Train Model

In [ ]:
model = YOLO(Path("checkpoints", "yolov8n-seg.pt"))

_ = model.train(
    data=Path("datasets", "SupermarketScanner-5", "data.yaml"),
    epochs=64,
    name="smktscnr",
)

In [ ]:
Image(Path("runs", "segment", "smktscnr", "results.png"))

In [ ]:
Image(Path("runs", "segment", "smktscnr", "confusion_matrix.png"))

In [ ]:
model.val(
    split="test",
    name="smktscnr_val",
)

#### Centralise Model

In [ ]:
shutil.copy2(
    Path("runs", "segment", "smktscnr", "weights", "best.pt"),
    Path("checkpoints", "smktscnr.pt"),
)